# Week-2 bulletin languages: fine-tune a translation model on one GPU

This notebook fine-tunes a multilingual translation model from English into the languages of
ICPAC's member states, so the weekly bulletin and the Copilot's answers can be given in them.
It runs the scripts in this folder (`prepare_data.py`, `evaluate.py`, `train.py`,
`translate.py`) step by step. See `docs/language_training.md` for the plan, the datasets and
their licences.

| Member state | Languages trained here | Human templates only |
|---|---|---|
| Burundi | Kirundi (rn), French (fr), Kiswahili (sw) | |
| Djibouti | French (fr), Arabic (ar), Somali (so) | Afar |
| Eritrea | Tigrinya (ti), Arabic (ar) | Tigre |
| Ethiopia | Amharic (am), Afaan Oromo (om), Tigrinya (ti), Somali (so) | Afar |
| Kenya | Kiswahili (sw) | |
| Rwanda | Kinyarwanda (rw), French (fr), Kiswahili (sw) | |
| Somalia | Somali (so), Arabic (ar) | |
| South Sudan | Arabic (ar) | Dinka, Nuer |
| Sudan | Arabic (ar) | |
| Tanzania | Kiswahili (sw) | |
| Uganda | Luganda (lg), Kiswahili (sw) | |

**Before you start**

* The GPU must have room. A full fine-tune of NLLB-600M needs about 18 GB; LoRA on the 1.3B
  model about 14 GB. The cell below stops if another job (for example a cGAN training run)
  holds the memory. On a shared workstation, run `run_when_gpu_free.sh` instead: it waits for
  the GPU and then runs the same steps.
* NLLB weights are CC-BY-NC-4.0 and so are the fine-tuned ones. Get ICPAC's sign-off, or set
  `MODEL = "google/madlad400-3b-mt"` with `LORA = True` (Apache-2.0).
* Forecasters' translations of `templates.en.tsv` go in `translations/templates.<lang>.tsv`.
  Without them the model learns only general text; with them it learns the bulletin's
  wording (they are repeated 20 times in training).

In [ ]:
# Settings: change these, then run every cell from the top.
LANGUAGES = ["sw", "am", "so", "om", "ti", "rw", "rn", "lg", "ar", "fr"]
MODEL = "facebook/nllb-200-distilled-600M"  # or "facebook/nllb-200-distilled-1.3B", "google/madlad400-3b-mt"
LORA = False  # True for the 1.3B and 3B models on a 32 GB card
EPOCHS = 2
BATCH = 16  # per step; 16 x 4 accumulation = 64 sentence pairs per update
ACCUMULATE = 4
MINED_ROWS = 150_000  # general sentence pairs per language (best-scoring first)
RUN = "runs/nllb600m"  # where the model is written
NEEDED_GB = 14 if LORA else 18

## 1. Check the GPU

In [ ]:
import shutil, subprocess


def gpu_memory():
    """(name, free GB, total GB, processes using it) of GPU 0."""
    out = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.free,memory.total",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.splitlines()[0]
    name, free, total = [part.strip() for part in out.split(",")]
    users = subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=pid,process_name,used_memory",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
    ).stdout.strip()
    return name, int(free) / 1024, int(total) / 1024, users


assert shutil.which("nvidia-smi"), "No NVIDIA driver found on this machine"
name, free, total, users = gpu_memory()
print(f"{name}: {free:.1f} GB free of {total:.1f} GB")
if users:
    print("In use by:\n" + users)
if free < NEEDED_GB:
    raise SystemExit(
        f"Only {free:.1f} GB free; this run needs about {NEEDED_GB} GB. Wait for the other job "
        "to finish (or run run_when_gpu_free.sh, which waits), or use the GH200 cluster."
    )

## 2. Install the training environment (once)

In [ ]:
# Match the PyTorch build to the CUDA driver (nvidia-smi shows its CUDA version).
# RTX 5000 Ada (x86_64): the cu124 or cu126 wheels. GH200 (aarch64): use NVIDIA's PyTorch
# container (nvcr.io/nvidia/pytorch) or the aarch64 cu126 wheels.
%pip install --quiet torch --index-url https://download.pytorch.org/whl/cu126
%pip install --quiet -r requirements.txt

In [ ]:
import torch

print(torch.__version__, "CUDA", torch.version.cuda, "bf16:", torch.cuda.is_bf16_supported())
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), (
    "This notebook trains in bf16 on a CUDA GPU"
)

## 3. Sign in to Hugging Face

Needed for the gated test sets (FLORES+, SALT): request access on their dataset pages first.

In [ ]:
from huggingface_hub import login

login()  # paste a read token from https://huggingface.co/settings/tokens

## 4. Prepare the data (about 10-20 minutes: downloads, then filtering)

In [ ]:
!python prepare_data.py --languages {" ".join(LANGUAGES)} --out data --templates translations --mined-rows {MINED_ROWS}

## 5. Score the untouched model (the baseline to beat)

In [ ]:
!python evaluate.py --model {MODEL} --data data --languages {" ".join(LANGUAGES)} | tee baseline.txt

## 6. Train

Times on one GPU, two epochs, 10 languages with 150,000 general pairs each (about 1.5
million pairs):

| Model | RTX 5000 Ada (32 GB) | One GH200 (96-144 GB) |
|---|---|---|
| NLLB-600M, full fine-tune | 5-7 hours | about 1 hour |
| NLLB-1.3B, LoRA | 7-10 hours | about 1.5 hours (or a full fine-tune in 2-3 hours) |
| MADLAD-3B, LoRA | 14-20 hours | 2.5-4 hours |

Halve the time with `MINED_ROWS = 75_000`; the forecasters' templates matter more than
the general data. Checkpoints are written every 2,000 steps, so a stopped run resumes.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
lora = "--lora" if LORA else ""
!python train.py --model {MODEL} {lora} --data data --out {RUN} --languages {" ".join(LANGUAGES)} --epochs {EPOCHS} --batch {BATCH} --accumulate {ACCUMULATE}

## 7. Score the fine-tuned model and compare

In [ ]:
!python evaluate.py --model {RUN}/final --data data --languages {" ".join(LANGUAGES)} | tee finetuned.txt
print(open("baseline.txt").read())

Accept the model only if, for every language: chrF++ is not below the baseline on the
news test, it is higher on the bulletin sentences, and `numbers_kept` is 1.0 (every number,
date and amount survives). Then native-speaker forecasters review a sample before any
translated bulletin is published; machine translations stay labelled as such.

## 8. Try a bulletin sentence

In [ ]:
sentence = "Heavy rainfall (above 200 mm) is expected over western Kenya and Uganda from 3 to 9 November 2026."
for lang in LANGUAGES:
    print(lang, end=": ", flush=True)
    !python translate.py --model {RUN}/final --lang {lang} "{sentence}"

## 9. Package for the web service

CTranslate2 int8 makes the model small and fast enough for a CPU replica of the forecast
service. Copy the folder to the server; the Copilot and the bulletin read it from
`TRANSLATION_MODEL_DIR` once ICPAC has reviewed it.

In [ ]:
%pip install --quiet ctranslate2
!ct2-transformers-converter --model {RUN}/final --output_dir {RUN}/ct2-int8 --quantization int8 --force
!du -sh {RUN}/ct2-int8